In [ ]:
from vizdoom import *
import cv2
import numpy as np
import time
import random
from matplotlib import pyplot as plt
import pandas as pd
import re
import pandas as pd
import csv
import os
from tqdm import tqdm

# Gym imports
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Discrete, Box

# Ollama imports
import requests

In [ ]:
# Stable Baselines3 imports
from stable_baselines3 import PPO
from stable_baselines3.common import env_checker
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback, CallbackList
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor

In [ ]:
# Paths
CONFIG_PATH = './github/ViZDoom/scenarios/take_cover.cfg'
LOG_DIR = './logs'

In [ ]:
class PerceptionLayer:
    URGENCY_IMMINENT   =  150
    URGENCY_APPROACHING = 400
    SIDE_THRESHOLD = 50
    URGENCY_ENC = {"DISTANT": 0, "APPROACHING": 1, "IMMINENT": 2}
    SIDE_ENC    = {"CENTER": 0, "LEFT": 1, "RIGHT": 2}
    DIR_ENC     = {"PARALLEL TO": 0, "AWAY FROM": 1, "TOWARD": 2}
    HIT_ENC     = {None: 0, "HIT": 1, "FATAL": 2}
    MAX_PROJ    = 3

    ACTION_NAMES = {0: "move_left", 1: "move_right"}

    def __init__(self):
        self.prev_health  = 100
        self.last_action  = None
        self.step_count   = 0

    def reset(self):
        self.prev_health = 100
        self.last_action = None
        self.step_count  = 0

    def update(self, state):
        self.step_count += 1

        player_y, health, hit = self._get_player_state(state)
        projectiles = self._get_projectiles(state, player_y)
        action = self.last_action
        text =  self._serialize(health, projectiles, hit, action)
        features = self._get_features(health, projectiles, hit, action)

        return text, features
    
    def update_action(self, action):
        self.last_action = action
    
    def _get_player_state(self, state):
        vars = state.game_variables
        health = vars[0]
        player_y = vars[2]

        delta_h = health - self.prev_health
        self.prev_health = health
        hit = self._detect_hit(health, delta_h)

        return player_y, health, hit
    
    def _detect_hit(self, health, delta_h):
        if health <= 0:
            return "FATAL"
        if delta_h < 0:
            return "HIT"
        return None
    
    def _get_projectiles(self, state, player_y):
        projectiles = []
        for label in state.labels:
            if label.object_name == "DoomImpBall":
                delta_y = label.object_position_y - player_y
                delta_x = label.object_position_x
                projectiles.append({
                "delta_y": delta_y,
                "delta_x": delta_x,
                "side":    self._side(delta_y),
                "urgency": self._urgency(delta_x),
                })
        
        # Returns projectiles sorted by urgency (closest to player first)
        return sorted(projectiles, key=lambda p: abs(p["delta_x"]))
    
    def _side(self, delta_y: float):
        if abs(delta_y) < self.SIDE_THRESHOLD:
            return "CENTER"
        return "LEFT" if delta_y < 0 else "RIGHT"

    def _urgency(self, delta_x: float):
        dist = abs(delta_x)
        if dist < self.URGENCY_IMMINENT:
            return "IMMINENT"
        elif dist < self.URGENCY_APPROACHING:
            return "APPROACHING"
        return "DISTANT"
    
    def _serialize(self, health, projectiles, hit=None, action=None):
        lines = []
        
        if hit == "fatal":
            lines.append("Health: 0/100 (DIED this step)")
        elif hit == "hit":
            lines.append(f"Health: {int(health)}/100 (first hit, critical condition)")
        else:
            lines.append(f"Health: {int(health)}/100")

        if not projectiles:
            lines.append("No projectiles visible.")
            if action is not None:
                lines.append(f"Action taken: {self.ACTION_NAMES.get(action, 'none')} (no threat present)")
        else:
            lines.append(f"{len(projectiles)} projectile(s) detected:")
            for i, p in enumerate(projectiles, 1):
                if action is not None:
                    direction = self._evaluate_direction(action, p["side"])
                    lines.append(f"  {i}. {p['urgency'].upper()} threat on the {p['side']} side. The player is MOVING {direction} this threat")
                else:
                    lines.append(f"  {i}. {p['urgency'].upper()} threat on the {p['side']} side")

        return "\n".join(lines)

    def _evaluate_direction(self, action, threat_side):
        if threat_side == "center":
            return "PARALLEL TO"
        if (action == 0 and threat_side == "right") or (action == 1 and threat_side == "left"):
            return "AWAY FROM"
        return "TOWARD"
    
    def _get_features(self, health, projectiles, hit, action):
        health_norm  = round(health / 100.0, 4)
        hit_status   = self.HIT_ENC[hit]
        num_proj     = len(projectiles)
        action_enc   = action if action is not None else -1

        proj_features = []
        for i in range(self.MAX_PROJ):
            if i < len(projectiles):
                p = projectiles[i]
                direction = self._evaluate_direction(action, p["side"]) if action is not None else "PARALLEL TO"
                proj_features += [
                    1,                              
                    self.URGENCY_ENC[p["urgency"]], 
                    self.SIDE_ENC[p["side"]],       
                    self.DIR_ENC[direction],        
                ]
            else:
                proj_features += [0, 0, 0, 0]

        features = tuple([health_norm, hit_status, num_proj, action_enc] + proj_features)

        return features

In [ ]:
class LLMAgent:
    ACTION_NAMES = {0: "move_left", 1: "move_right"}

    def __init__(self, model_name, backend_url):
        self.backend_url = backend_url
        self.model_name = model_name
        self.parse_failures = 0
        self._cache = {}
        self.payload = {
            "model" : self.model_name,
            "system" : self._build_system_prompt(),
            "prompt" : "",
            "stream" : False,
        }

    def evaluate(self, state_text, action_taken):
        prompt  = self._build_prompt(state_text)
        response = self._call_llm(prompt)
        reward = self._parse_reward(response)
        
        return reward
    
    def _build_system_prompt(self):
        return """You are a reward model for a dodge game. Evaluate the player's action and return a reward score.
        ### SCORING CRITERIA:
        - Player moving AWAY from IMMINENT threat:    +1.0
        - Player moving TOWARD IMMINENT threat:       -1.0
        - Player moving AWAY from APPROACHING threat: +0.5
        - Player moving TOWARD APPROACHING threat:    -0.5
        - No threats:                                 +0.0

        ### FORMULA:
        TOTAL = mean(threat scores)

        ### STEP BY STEP:
        1. For each projectile, check if player is moving toward or away and assign score
        2. Compute mean of all threat scores

        ### OUTPUT FORMAT:
        Write the final line exactly as: REWARD: <number>"""
    
    def _build_prompt(self, state_text):
        return f"""### NOW EVALUATE: {state_text}"""

    def _call_llm(self, prompt):
        self.payload["prompt"] = prompt

        try:
            response = requests.post(self.backend_url, json= self.payload)

            # Check response status
            if response.status_code == 200:
                return response.json()["response"]
            else:
                print(f"Ollama returned error: {response.status_code}")
                return ""
        except requests.exceptions.ConnectionError:
            print("Failed to connect to Ollama backend.")
            return ""   
    
    def _parse_reward(self, response):
        try:
            match = re.search(r'REWARD:\s*([+-]?\d+\.?\d*)', response)
            if match:
                return max(-5.0, min(5.0, float(match.group(1))))
            self.parse_failures += 1
            return 0.0 # Fallback if no number found
        except ValueError:
            self.parse_failures += 1
            return 0.0 # Fallback 

In [ ]:
# Create Vizdoom OpenAI Gym Environment
class VizDoomGym(Env): 
    def __init__(self, render=False): 
        # Setup the game 
        super().__init__()
        self.frame_skip = 4
        self.game = DoomGame()
        self.game.load_config(CONFIG_PATH)
        
        # Render frame logic
        self.game.set_window_visible(render)
        
        # Start the game 
        self.game.init()
        
        # Create the action space and observation space
        self.observation_space = Box(low=0, high=255, shape=(100,160,1), dtype=np.uint8) 
        self.action_space = Discrete(2) # Move left, move right
        self._actions = np.eye(2, dtype=np.uint8)
        
    # This is how we take a step in the environment
    def step(self, action):
        # Specify action and take step 
        reward = self.game.make_action(self._actions[action].tolist(), self.frame_skip) 
        
        # Get the new state of the game and check if it's done
        if self.game.get_state(): 
            obs    = self._process_frame(self.game.get_state().screen_buffer)
            health = self.game.get_state().game_variables[0]
            info   = {"health": health}
        else: 
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {"health": 0}
        
        terminated = self.game.is_episode_finished()
        truncated = False

        return obs, reward, terminated, truncated, info
    
    # Define how to render the game or environment 
    def render(self): 
        pass
    
    # Starting a new game 
    def reset(self, seed=None, options=None): 
        super().reset(seed=seed)
        self.game.new_episode()
        state = self.game.get_state()

        if state is not None:
            obs = self._process_frame(state.screen_buffer)
        else:
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)

        return obs, {}
    
    # Call to close down the game
    def close(self): 
        self.game.close()

    def _process_frame(self, buffer: np.ndarray):
        hwc  = np.moveaxis(buffer, 0, -1)                           
        gray = cv2.cvtColor(hwc, cv2.COLOR_RGB2GRAY)               
        resized = cv2.resize(gray, (160, 100), interpolation=cv2.INTER_CUBIC)
        clipped = np.clip(resized, 0, 255).astype(np.uint8)         
        return clipped.reshape(100, 160, 1)

### Extracting states from real-world instances

In [ ]:
env = VizDoomGym(render=False)
perception = PerceptionLayer()
agent = LLMAgent(model_name="llama3.2:3b", backend_url="http://localhost:11434/api/generate")
episodes = 10

FEATURE_COLS = [
    "health", "hit_status", "num_proj", "action",
    "p0_present", "p0_urgency", "p0_side", "p0_direction",
    "p1_present", "p1_urgency", "p1_side", "p1_direction",
    "p2_present", "p2_urgency", "p2_side", "p2_direction",
]

csv_path = os.path.join(LOG_DIR, f"game_states.csv")
txt_path = os.path.join(LOG_DIR, f"game_states.txt")
csv_columns = ["text", "episode", "step", "reward_base", "reward_llm"] + FEATURE_COLS

with open(csv_path, "w", newline="") as csv_file, open(txt_path, "w") as txt_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_columns)
    writer.writeheader()

    for episode in tqdm(range(episodes)):
        obs, info = env.reset()
        perception.reset()
        truncated  = False
        terminated = False
        step = 0

        while not terminated and not truncated:
            #!DEBUG
            print("GAME START")

            step += 1
            state = env.game.get_state()
            state_text, features = perception.update(state)

            action = random.randint(0, 1)
            #!DEBUG
            print(f"action : {action}")

            llm_reward = agent.evaluate(state_text, action)

            obs, reward, terminated, truncated, info = env.step(action)
            perception.update_action(action)

            txt_file.write(f"{'='*60}\n")
            txt_file.write(f"Episode {episode:04d} | Step {step:04d}\n")
            txt_file.write(f"{'-'*60}\n")
            txt_file.write(state_text + "\n")
            txt_file.write(f"Reward base : {reward:.4f}\n")
            txt_file.write(f"Reward LLM  : {llm_reward:.4f}\n")
            txt_file.write("\n")

            row = {
                "episode":     episode,
                "step":        step,
                "reward_base": reward,
                "reward_llm":  llm_reward,
            }
            row.update(dict(zip(FEATURE_COLS, features)))
            writer.writerow(row)

env.close()
print(f"Log salvati in:\n  {csv_path}\n  {txt_path}")